Desarrollo código pregunta 1

In [5]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [51]:
import cv2
from google.colab.patches import cv2_imshow
import matplotlib
def interpolacion_puntos_control(h,puntos):
  #puntos -> puntos = [(0, 0), (60, 0.5),(hi,mi)]
  Ordenados = sorted(puntos, key=lambda x: x[0])
  largo = len(Ordenados) #cantidad de par de puntos de control

  for i in range (largo-1):
    par_inferior = Ordenados[i][0]
    par_superior= Ordenados[i+1][0]

    if par_inferior <= h <= par_superior and (Ordenados[i+1][0]-Ordenados[i][0] != 0):

      m = Ordenados[i][1] + (h - Ordenados[i][0])/(Ordenados[i+1][0]-Ordenados[i][0])*(Ordenados[i+1][1]-Ordenados[i][1])
      return m

  par_ultimo= Ordenados[-1] #(340,0)
  par_primero = Ordenados[0] #(10,0) (hi,mi)

  if h < par_primero[0]:
    h = h+360 #le doy una vuelta completa
    m = par_ultimo[1] + (h - par_ultimo[0])/((par_primero[0]+360)-par_ultimo[0])*(par_primero[1]-par_ultimo[1])
    return m

  elif h > par_ultimo[0]:
    m = par_ultimo[1] + (h - par_ultimo[0])/(par_primero[0]+360-par_ultimo[0])*(par_primero[1]-par_ultimo[1])
    return m


def ColorSaturation(imagen_en_rgb, puntos, modo):

    imagen_en_rgb = imagen_en_rgb.astype(np.float32)
    imagen_en_rgb = imagen_en_rgb/ 255.0
    # se normalizan los valores de pixeles, quedan como (ppt profe)
    # Los valores de R G B se dividen por 255 para cambiar su rango
    # de 0,255 a 0,1
    if modo == "HS":
      imagen_en_hsv = cv2.cvtColor(imagen_en_rgb, cv2.COLOR_RGB2HSV)
      print(f"PRUEBA 1: Pixel HS {imagen_en_hsv[698][450]}")
      f_c = imagen_en_hsv.shape #entrega (fila , columnas, canales)
      #print(imagen_en_hsv[0][0]) se obtuvo [ 14 117 157] que seria [H S V]

      matrix_para_m=np.zeros((f_c[0],f_c[1]))
      for i in range (f_c[0]):
        for j in range(f_c[1]):
          h = imagen_en_hsv[i][j][0]
          m = interpolacion_puntos_control(h,puntos)
          matrix_para_m[i][j] = m

      for a in range(f_c[0]):
        for b in range (f_c[1]):
          s = imagen_en_hsv[a][b][1] #el valor del medio
          m_ab= matrix_para_m[a][b]
          g_m = 1 + m_ab
          S_prima = g_m * s
          # hasta aqui tendría valores H en grados, S y V en 0,1
          # un valor neutral que no alteraría el s para pasar a S prima
          # sería m = 0 -> s_prima = 1 * s
          if 1 < S_prima:
            S_prima = 1
          elif S_prima < 0:
            S_prima=0
          imagen_en_hsv[a][b][1] = S_prima
      print(f"PRUEBA 2: Imagen con S' {imagen_en_hsv[698][450]}")
      imagen_devuelta_rgb= cv2.cvtColor(imagen_en_hsv, cv2.COLOR_HSV2RGB)
      imagen_final = (imagen_devuelta_rgb * 255).astype(np.uint8)
      return imagen_final

    if modo == "CIE L*c*h":
      imagen_en_lab=cv2.cvtColor(imagen_en_rgb, cv2.COLOR_RGB2LAB)
      f_c2= imagen_en_lab.shape #arroja (3000,2000,3)

      for i in range(f_c2[0]):
        for j in range(f_c2[1]):
            a = imagen_en_lab[i][j][1]
            b = imagen_en_lab[i][j][2]
            imagen_en_lab[i][j][1] = np.sqrt(a**2 + b**2) # C
            imagen_en_lab[i][j][2] = np.degrees(np.arctan2(b, a)) %360 #h

      imagen_en_lch = imagen_en_lab.copy()
      matrix_para_c= np.zeros((f_c2[0],f_c2[1]))
      for a in range(f_c2[0]):
        for b in range (f_c2[1]):
          h = imagen_en_lch[a][b][2]
          m = interpolacion_puntos_control(h,puntos)
          matrix_para_c[a][b] = m

      for e in range(f_c2[0]):
        for v in range (f_c2[1]):
          c = imagen_en_lch[e][v][1] #el valor del medio
          m_ev= matrix_para_c[e][v]
          g_m = 1 + m_ev
          c_asterisco = g_m * c
          if c_asterisco < 0:
            c_asterisco = 0
          imagen_en_lch[e][v][1] = c_asterisco
      #tendría ya la matriz de pixeles en LCH con el valor de c nuevo
      # Volvemos desde L*C*h* a L*a*b*
      for e in range(f_c2[0]):
          for v in range(f_c2[1]):

              L = imagen_en_lch[e][v][0]
              c_asterisco = imagen_en_lch[e][v][1]
              h = imagen_en_lch[e][v][2]

              h_rad = np.radians(h)

              a_asterisco = c_asterisco * np.cos(h_rad)
              b_asterisco = c_asterisco * np.sin(h_rad)

              imagen_en_lab[e][v][0] = L
              imagen_en_lab[e][v][1] = a_asterisco
              imagen_en_lab[e][v][2] = b_asterisco

      imagen_devuelta_rgb = cv2.cvtColor(imagen_en_lab,cv2.COLOR_LAB2RGB)
      imagen_devuelta_rgb = np.clip(imagen_devuelta_rgb, 0, 1)
      imagen_final = (imagen_devuelta_rgb * 255).astype(np.uint8)
      return imagen_final


imagen = cv2.imread("/content/bts1.jpeg")
imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

#puntos =[(0,0), (10,0), (25,8.0), (40,0), (360,0)]
#resultado = ColorSaturation(imagen, puntos, "CIE L*c*h")
#cv2_imshow(cv2.cvtColor(resultado, cv2.COLOR_RGB2BGR))


Desarrollo pregunta 2

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

def ecualizacion_local(imagen, tamano_region, separacion, contraste,nro_bins):

    fila_col = imagen.shape
    f = fila_col[0]
    c = fila_col[1]
    transformaciones = []
    for fila in range(0, f, separacion):
        for columna in range(0, c, separacion):

            fila_fin = min(fila + tamano_region, f)
            columna_fin = min(columna + tamano_region, c)

            region = imagen[fila:fila_fin, columna:columna_fin]
            histograma = np.histogram(region, bins=nro_bins, range=(0, 256))[0]
            N = region.size
            if contraste > 0:
              limite = contraste * (N / nro_bins)
              exceso = np.sum(np.maximum(histograma - limite, 0))
              histograma = np.minimum(histograma, limite)
              histograma = np.round(histograma + exceso / nro_bins)
            cdf = np.cumsum(histograma)
            valor_cdf_no_cero = cdf[cdf > 0]

            if len(valor_cdf_no_cero) > 0:
                CDF_min = valor_cdf_no_cero[0]
            else:
                CDF_min = 0

            matriz_transformacion = np.zeros(256)

            if N > CDF_min:
                for k in range(256):
                    bin_k = min(k * nro_bins // 256, nro_bins - 1)
                    if cdf[bin_k] > 0:
                        matriz_transformacion[k] = ((cdf[bin_k] - CDF_min) /(N - CDF_min)) * 255
            else:
              matriz_transformacion = np.arange(256)
            transformacion = np.round(matriz_transformacion).astype(np.uint8)
            transformaciones.append(transformacion)

    suma = np.zeros_like(imagen,dtype=np.float32)
    contador = np.zeros_like(imagen,dtype=np.float32)
    indice = 0
    for fila in range(0, f, separacion):
        for columna in range(0, c, separacion):

            fila_fin = min(fila + tamano_region, f)
            columna_fin = min(columna + tamano_region, c)
            region = imagen[fila:fila_fin, columna:columna_fin]

            transformacion = transformaciones[indice]

            region_ecualizada = transformacion[region]

            suma[fila:fila_fin, columna:columna_fin] += region_ecualizada
            contador[fila:fila_fin, columna:columna_fin] += 1
            indice += 1

    imagen_ecualizada = np.round(suma / contador).astype(np.uint8)
    return imagen_ecualizada


imagen = cv2.imread("/content/P2_IMG_2423.tif")
imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

#transformaciones = ecualizacion_local(imagen, 1, 25, 20, 256)
#cv2_imshow(cv2.cvtColor(transformaciones, cv2.COLOR_GRAY2RGB))




Desarrollo pregunta 3

In [50]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

def reescalado_interpolacionbi(imagen, factor, modo):

  alto = imagen.shape[0]
  ancho = imagen.shape[1]

  alto_out= round(alto*factor)
  ancho_out= round(ancho*factor)
  dimensiones = len(imagen.shape)
  imagen_float = imagen.astype(np.float32)
  if dimensiones == 3:
    #RGB
    matriz_salida = np.zeros((alto_out,ancho_out,imagen.shape[2]),dtype= np.float32)
  elif dimensiones == 2:
    #grises
    matriz_salida = np.zeros((alto_out,ancho_out),dtype= np.float32)


  for i in range(alto_out):
    for j in range(ancho_out):

      posicion_x_original = j/factor
      posicion_y_original = i/factor

      if modo == "vecino_mas_cercano":
        indice_i = np.fix(posicion_y_original) #capsula
        indice_j= np.fix(posicion_x_original)

        indice_i = int(np.minimum(alto-1, np.maximum(indice_i, 0)))
        indice_j = int(np.minimum(ancho-1, np.maximum(indice_j, 0)))

        matriz_salida[i, j] = imagen_float[indice_i, indice_j]
      else:
        #bilineal, visualiza una rejilla de 9 espacios, sacas las 4 esquinas
        x0= int(np.fix(posicion_x_original)) #buscamos indices
        y0= int(np.fix(posicion_y_original))

        x1= x0+1
        y1= y0+1

        x0_recort = int(np.minimum(ancho-1, np.maximum(x0, 0)))
        x1_recort = int(np.minimum(ancho-1, np.maximum(x1, 0)))
        y0_recort = int(np.minimum(alto-1, np.maximum(y0, 0)))
        y1_recort = int(np.minimum(alto-1, np.maximum(y1, 0)))

        #Utilizando las formulas dadas en los ppts:

        f11 = imagen_float[y0_recort, x0_recort]   # f(x1,y1)
        f12 = imagen_float[y1_recort, x0_recort]   # f(x1,y2)
        f21 = imagen_float[y0_recort, x1_recort]   # f(x2,y1)
        f22 = imagen_float[y1_recort, x1_recort]   # f(x2,y2)

        fy1 = f11 + (f21 - f11) * (posicion_x_original - x0)
        fy2 = f12 + (f22 - f12) * (posicion_x_original - x0)

        valor = fy1 + (fy2 - fy1) * (posicion_y_original - y0)

        matriz_salida[i, j] = valor

  matriz_salida = np.round(matriz_salida)
  matriz_salida = np.clip(matriz_salida, 0, 255)
  matriz_salida = matriz_salida.astype(np.uint8)
  return matriz_salida


#imagen3= cv2.imread("/content/P3_IMG_2387_crop.tif")
#imagen3 = cv2.cvtColor(imagen3, cv2.COLOR_BGR2GRAY)
#imagen4 = cv2.cvtColor(imagen3, cv2.COLOR_BGR2RGB)

#resultado_vecino = reescalado_interpolacionbi(imagen3, 1.5, "vecino_mas_cercano")
#resultado_bilineal = reescalado_interpolacionbi(imagen4, 1.5, "bilineal")